In [26]:
import os
import json
import re
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
import numpy as np
import pprint

from typing import List, Dict, Any, Tuple
from collections import defaultdict, Counter
from utils import count_relaxed_matches, count_exact_matches, exact_match_results, relax_match_results, sample_to_prediction_list


In [27]:
# model_folder = "Output_llama3.1_8b_no_sim"
# model_folder = "Output_mistral-small_24b_no_sim"
# model_folder = "Output_gemma4_26b_no_sim"
# model_folder = "Output_gemma4_31b_no_sim"
model_folder = "Output_nemotron3_30b_no_sim"

In [ ]:
if os.path.exists(f"results/Output_verification_em.csv") == True:
    os.remove(f"results/Output_verification_em.csv")
if os.path.exists(f"results/Output_verification_rm.csv") == True:
    os.remove(f"results/Output_verification_rm.csv")


tp_rm, fp_rm, fn_rm, od_rm = 0, 0, 0, 0
tp_em, fp_em, fn_em, od_em = 0, 0, 0, 0
for sample in os.listdir(model_folder):
    # if sample in ['sample_68', 'sample_635', 'sample_71', 'sample_324','sample_2748']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict_processed = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    preds_dict_processed.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                
        #Deduplication
        preds_dict_final = []
        seen = set()
        for d in preds_dict_processed:
            t = tuple(sorted(d.items()))
            if t not in seen:
                seen.add(t)
                preds_dict_final.append(d)
                
        ## Evaluation
        em, em_mm, em_ud, em_od = exact_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
        rm, rm_mm, rm_ud, rm_od = relax_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
    
        tp_rm+= rm
        fp_rm+= rm_mm
        fn_rm+= rm_ud
        od_rm+= rm_od

            
        tp_em+= em
        fp_em+= em_mm
        fn_em+= em_ud
        od_em+= em_od

rm_precision = tp_rm/(tp_rm + fp_rm + od_rm)
rm_recall = tp_rm/(tp_rm + fn_rm)

em_precision = tp_em/(tp_em + fp_em + od_em)
em_recall = tp_em/(tp_em + fn_em)

print(f"SUMMARY EXACT MATCH: TP {tp_em}, FP {fp_em}, UD {fn_em}, OD {od_em}")
print(f"MICRO AVERAGE SCORE EXACT MATCH: Precision: {em_precision}, Recall: {em_recall}, F1 score: {2*em_precision*em_recall/(em_precision + em_recall)}")

print(f"SUMMARY RELAX MATCH: TP {tp_rm}, FP {fp_rm}, UD {fn_rm}, OD {od_rm}")
print(f"MICRO AVERAGE SCORE RELAX MATCH: Precision: {rm_precision}, Recall: {rm_recall}, F1 score: {2*rm_precision*rm_recall/(rm_precision + rm_recall)}")

evaluating sample_38_20210610044413_Hayden:
[{'abdominal pain': 'PROBLEM'}, {'30-pound weight loss': 'PROBLEM'}, {'jaundice': 'PROBLEM'}, {'epigastric pain': 'PROBLEM'}, {'CT scan': 'TEST'}, {'pancreatic mass': 'PROBLEM'}, {'lymph nodes': 'PROBLEM'}, {'liver metastases': 'PROBLEM'}, {'abdominal pain': 'PROBLEM'}, {'30 - pound weight loss': 'PROBLEM'}, {'jaundice': 'PROBLEM'}, {'epigastric pain': 'PROBLEM'}, {'4x3x2 cm pancreatic mass': 'PROBLEM'}, {'involved lymph nodes': 'PROBLEM'}, {'ring enhancing lesions': 'PROBLEM'}, {'liver metastases': 'PROBLEM'}, {'questionable pseudocyst': 'PROBLEM'}, {'ERCP': 'TEST'}, {'stent': 'TREATMENT'}, {'strictured pancreatic duct': 'PROBLEM'}, {'strictured bile duct': 'PROBLEM'}, {'10-frenchx9 cm stent': 'TREATMENT'}, {'bilirubin': 'PROBLEM'}, {'liver tests': 'TEST'}, {'nausea': 'PROBLEM'}, {'Zofran': 'DRUG'}, {'Phenergan': 'DRUG'}, {'stent placement': 'TREATMENT'}, {'nausea': 'PROBLEM'}, {'white count': 'TEST'}, {'hemoglobin': 'TEST'}, {'hematocrit': 

In [29]:
df = pd.read_csv('results/Output_verification_rm.csv',header=None)
df.columns = ['file','RM','MM','UD','OD','Prec','Sens','F1']
print(df['F1'].mean())

df.sort_values(by='OD',ascending=False).head(10)

0.7419070422535211


,file,RM,MM,UD,OD,Prec,Sens,F1
26,sample_2748,134,3,7,93,0.5826,0.9504,0.7224
83,sample_635,138,54,12,87,0.4946,0.9200,0.6434
13,sample_128_20210112211711,74,12,8,82,0.4405,0.9024,0.5920
18,sample_30_20210610044412_Isra,63,5,6,78,0.4315,0.9130,0.5860
16,sample_2762,178,45,14,75,0.5973,0.9271,0.7265
25,sample_58_20210604143820_Isra,177,73,7,71,0.5514,0.9620,0.7010
63,sample_1714,42,31,8,70,0.2937,0.8400,0.4352
125,sample_47_20210604143818_Isra,119,18,31,69,0.5777,0.7933,0.6685
52,sample_86,73,18,3,68,0.4591,0.9605,0.6213
81,sample_641,179,73,14,65,0.5647,0.9275,0.7020


# Further analysis